# 把 NES-VMC 的列旋转回本征态：如何计算 $v$

背景：NES-VMC 的 K 个列 $\psi_j$ 只是 K 维本征子空间的**任意基向量**（GL(K) 规范自由），并不等于本征函数。
要得到真正的本征态，需要在 K 列张成的子空间上求解**广义本征问题**，再把列旋转过去：

$$
M\,v = \lambda\,S\,v,\qquad M_{ij}=\langle\psi_i|H|\psi_j\rangle,\quad S_{ij}=\langle\psi_i|\psi_j\rangle
$$

旋转后的列：
$$
\psi' = \psi\,v \quad\Longleftrightarrow\quad \varphi_k(x)=\sum_j\psi_j(x)\,v_{kj}
$$

旋转后第 $k$ 列就是第 $k$ 个（按能量升序的）本征态，彼此正交，且 $\langle\varphi_k|H|\varphi_k\rangle\approx\lambda_k$。

本 notebook 用 H2 / 6-31G / K=4 训练好的参数完整演示这一过程，并逐条验证。

## 1. 载入环境与训练参数

In [ ]:
import pickle
import numpy as np
import flax.nnx as nnx
from scipy.linalg import eigh

from NES_VMC_V1 import NESTotalAnsatz
from H2_631G import SINGLE_SIZE, ha, hi, K, E_fcis

HISTORY_FILE = './data/26-09-08-18-09_history_natural_gradient_H2_molecule_K4.pkl'
with open(HISTORY_FILE, 'rb') as f:
    history = pickle.load(f)

print(f'K = {K}, 构型数 = {hi.n_states}, FCI 能级 = {np.round(E_fcis, 4)}')

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: Debug multi-node HPC? `djaxrun -np 2 python Examples/Sharding/multi_process.py`

H2 分子基本信息
HF energy = -0.99749729 Ha
Total electrons = (1, 1)
Total basis functions = 4
H₂ FCI 基准能量
E0 = -1.05434745 Ha  |  激发能: 0.0000 eV
E1 = -0.95790573 Ha  |  激发能: 2.6243 eV
E2 = -0.66895227 Ha  |  激发能: 10.4871 eV
E3 = -0.55140192 Ha  |  激发能: 13.6859 eV



HF reference state: [0 0 0 1 0 0 0 1]


single_edges: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (4, 5), (4, 6), (4, 7), (5, 6), (5, 7), (6, 7)]


K = 4, 构型数 = 16, FCI 能级 = [-1.0543 -0.9579 -0.669  -0.5514]


In [2]:
# 全部单态构型上的哈密顿量矩阵
x = np.asarray(hi.all_states(), dtype=np.complex64)   # (16, n_spin)
H = np.asarray(ha.to_dense(), dtype=np.complex128)    # (16, 16)

# 用与训练同构的总拟设拿 graphdef
total_ansatz = NESTotalAnsatz(
    n_spin_orbitals=SINGLE_SIZE, n_states=K, hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
)
graphdefs = [nnx.split(total_ansatz.single_ansatz_list[j])[0] for j in range(K)]

params = history['params'][-1]

# 每列在所有构型上的对数振幅 -> exp -> 振幅
logpsi = [nnx.merge(graphdefs[j], params['single_ansatz_list'][j])(x) for j in range(K)]
cols = [np.exp(np.asarray(lp, dtype=np.complex128)) for lp in logpsi]
Psi = np.stack(cols, axis=1)    # (16, K) 原始列

print('原始列矩阵 Psi 形状:', Psi.shape)

原始列矩阵 Psi 形状: (16, 4)


## 2. 构造 M、S 矩阵

In [3]:
HPsi = H @ Psi
M = Psi.conj().T @ HPsi           # M_ij = <psi_i|H|psi_j>
S = Psi.conj().T @ Psi            # S_ij = <psi_i|psi_j>
# 强制 Hermitian（去掉浮点误差）：注意必须用 .conj().T
M = 0.5 * (M + M.conj().T)
S = 0.5 * (S + S.conj().T)

print('M (实部):')
print(np.round(np.real(M), 3))
print('\nS 的归一化重叠 |S_ij|/sqrt(S_ii S_jj)（理想本征基应≈单位阵）:')
n_overlap = abs(S) / np.sqrt(np.diag(S)[:, None] * np.diag(S)[None, :])
print(np.round(n_overlap, 3))

M (实部):
[[ -62186.706  -46876.983  -18219.411  -79046.764]
 [ -46876.983  -48523.883  -16079.791  -58483.539]
 [ -18219.411  -16079.791   -6574.019  -26926.998]
 [ -79046.764  -58483.539  -26926.998 -127760.594]]

S 的归一化重叠 |S_ij|/sqrt(S_ii S_jj)（理想本征基应≈单位阵）:
[[1.   +0.j 0.893+0.j 0.866+0.j 0.901+0.j]
 [0.893+0.j 1.   +0.j 0.957+0.j 0.978+0.j]
 [0.866+0.j 0.957+0.j 1.   +0.j 0.975+0.j]
 [0.901+0.j 0.978+0.j 0.975+0.j 1.   +0.j]]


## 3. 求解广义本征问题 $Mv=\lambda Sv$：这就是「转回本征态」的 $v$

用 `scipy.linalg.eigh(M, S)` 一次解出本征值 $\lambda$ 与（$S$-正交归一）的系数矩阵 $v$。按能量升序排好。

In [4]:
lam, v = eigh(M, S)
order = np.argsort(lam.real)          # 能量升序
lam, v = lam[order], v[:, order]      # 列按能量升序重排

print('广义本征值 λ =', np.round(lam.real, 6))
print('FCI 本征值   =', np.round(E_fcis, 6))
print('\n旋转系数矩阵 v（φ_k = Σ_j ψ_j v[k,j]，行=能级升序，列=原始列）:')
print(np.round(np.real(v), 4))

广义本征值 λ = [-1.045074 -0.956903 -0.66358  -0.548844]
FCI 本征值   = [-1.054347 -0.957906 -0.668952 -0.551402]

旋转系数矩阵 v（φ_k = Σ_j ψ_j v[k,j]，行=能级升序，列=原始列）:
[[ 0.0027  0.0011  0.009   0.0095]
 [-0.0007 -0.0069 -0.0056 -0.0116]
 [ 0.0101 -0.0075 -0.0053  0.0248]
 [-0.0004 -0.0048 -0.0005 -0.0021]]


## 4. 旋转：把列变换成真正的本征态 $\varphi = \psi v$

In [5]:
# v 的第一维是「能级」，要对列（Psi 的第 2 维）作用：varphi = sum_j Psi[:,j] v[c,j]
Phi = Psi @ v              # (16, K) 本征态列

# 每个本征态的能量 = <phi|H|phi>/<phi|phi>
E_rot = np.diag((Phi.conj().T @ (H @ Phi)).real) / np.diag((Phi.conj().T @ Phi).real)

print('旋转后下列能量 :', np.round(E_rot, 6))
print('广义本征值 λ   :', np.round(lam.real, 6))
print('FCI 本征值     :', np.round(E_fcis, 6))

旋转后下列能量 : [-1.045074 -0.956903 -0.66358  -0.548844]
广义本征值 λ   : [-1.045074 -0.956903 -0.66358  -0.548844]
FCI 本征值     : [-1.054347 -0.957906 -0.668952 -0.551402]


## 5. 验证四件事

**(1) 旋转后每列能量 = 广义本征值**（旋转列确是本征态）

**(2) 旋转后列彼此正交**（重叠矩阵≈单位阵）

In [6]:
S_rot = Phi.conj().T @ Phi
S_rot_norm = abs(S_rot) / np.sqrt(np.diag(S_rot)[:, None] * np.diag(S_rot)[None, :])
print('旋转后列间归一化重叠（应≈单位阵，即正交）:')
print(np.round(S_rot_norm, 6))

print('\n原始列间重叠（对照，应非正交）:')
print(np.round(abs(S) / np.sqrt(np.diag(S)[:, None] * np.diag(S)[None, :]), 3))

旋转后列间归一化重叠（应≈单位阵，即正交）:
[[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]]

原始列间重叠（对照，应非正交）:
[[1.   +0.j 0.893+0.j 0.866+0.j 0.901+0.j]
 [0.893+0.j 1.   +0.j 0.957+0.j 0.978+0.j]
 [0.866+0.j 0.957+0.j 1.   +0.j 0.975+0.j]
 [0.901+0.j 0.978+0.j 0.975+0.j 1.   +0.j]]


**(3) 旋转后列与 FCI 本征态逐一一一对应**（重叠矩阵近似单位阵）

In [7]:
_, ef = np.linalg.eigh(H)                      # ef 的列 = FCI 本征态
# 先归一化旋转列，避免全局范数影响
Phi_n = Phi / np.sqrt(np.diag(Phi.conj().T @ Phi))[None, :]
overlap = np.abs(Phi_n.conj().T @ ef) ** 2     # 行=旋转列, 列=FCI 态

print('旋转列 vs FCI 本征态 |重叠|²（行=旋转列 0..3，列=FCI 态 0..3，应近似单位阵）:')
print(np.round(overlap[:, :4], 4))

旋转列 vs FCI 本征态 |重叠|²（行=旋转列 0..3，列=FCI 态 0..3，应近似单位阵）:
[[9.947e-01 1.000e-04 0.000e+00 0.000e+00]
 [1.000e-04 9.992e-01 0.000e+00 0.000e+00]
 [0.000e+00 0.000e+00 9.960e-01 0.000e+00]
 [0.000e+00 0.000e+00 0.000e+00 9.976e-01]]


**(4) 原始列为何不是本征态**：它们只是子空间的任意基，必须做上面的旋转才有「列↔能级」一一对应。

In [8]:
# 原始列归一化后与 FCI 的重叠 —— 每行最大值都落在基态，且分散于多态
Psi_n = Psi / np.sqrt(np.diag(Psi.conj().T @ Psi))[None, :]
ov_raw = np.abs(Psi_n.conj().T @ ef) ** 2
print('原始列 vs FCI |重叠|²（行=原始列 0..3，列=FCI 态 0..3）：')
print(np.round(ov_raw[:, :4], 3))

原始列 vs FCI |重叠|²（行=原始列 0..3，列=FCI 态 0..3）：
[[0.735 0.    0.194 0.066]
 [0.913 0.007 0.007 0.067]
 [0.864 0.003 0.076 0.051]
 [0.893 0.014 0.019 0.069]]


## 6. 结论

计算 $v$ 的方法就是解**广义本征问题** `scipy.linalg.eigh(M, S)`，其中 $M_{ij}=\langle\psi_i|H|\psi_j\rangle,\\ S_{ij}=\langle\psi_i|\psi_j\rangle$。旋转 $\varphi_k=\sum_j\psi_j v_{kj}$ 后：

- 第 $k$ 列即是第 $k$ 个（能量升序）本征态，$\langle\varphi_k|H|\varphi_k\rangle = \lambda_k$；
- 列彼此正交，且与 FCI 本征态一一对应；
- 这就是**分层冻结前必须做的规范固定**：冻结「旋转后的本征态列」，而非原始列。